# Query generation

## What this notebook does

This notebook shows how Retrieval-Augmented Generation can move beyond a single vector store workflow. It starts by enriching document chunks with metadata, then demonstrates how user questions can be converted into the right query form for different backends: metadata-aware vector retrieval, SQL generation for relational data, and routing across multiple retrievers.

The second half of the notebook connects these pieces into a larger RAG architecture. Instead of treating retrieval as a single fixed step, it shows how to choose the right data source, generate backend-specific queries, and apply postprocessing techniques such as RAG Fusion to improve the final context passed to the model.


In [1]:
# Select the LLM provider for the notebook: "openai", "ollama", or "gemini".
# Values are loaded from the project-root .env file when present.
import os
import getpass
from pathlib import Path

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings


def load_project_env(filename=".env", override=True):
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        env_path = directory / filename
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if override or key not in os.environ:
                    os.environ[key] = value
            return env_path
    return None


PROJECT_ENV_PATH = load_project_env(override=True)

# LangSmith tracing is configured before any LangChain objects are created.
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "query-generation")
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "true")

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
OLLAMA_EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-flash-latest")
GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2-preview")

SUPPORTED_LLM_PROVIDERS = {"openai", "ollama", "gemini"}
if LLM_PROVIDER not in SUPPORTED_LLM_PROVIDERS:
    raise ValueError(f"Unsupported LLM_PROVIDER: {LLM_PROVIDER}. Use one of: {sorted(SUPPORTED_LLM_PROVIDERS)}")


def _get_api_key(*names):
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    return getpass.getpass(f"Enter your {names[0]}")


class GeminiEmbeddingsOneByOne:
    def __init__(self, embeddings):
        self.embeddings = embeddings

    def embed_documents(self, texts):
        # Gemini embedding models can return one vector per batch with this
        # LangChain version. Chroma expects one vector per document, so embed
        # each chunk separately.
        return [
            self.embeddings.embed_documents([text], batch_size=1)[0]
            for text in texts
        ]

    def embed_query(self, text):
        return self.embeddings.embed_query(text)


def ensure_ollama_model_available(model_name):
    import ollama

    try:
        installed_models = {model.model for model in ollama.list().models}
    except Exception as exc:
        raise RuntimeError(
            f"Could not connect to Ollama at {OLLAMA_BASE_URL}. "
            "Start Ollama before running this notebook."
        ) from exc

    model_aliases = {name.removesuffix(":latest") for name in installed_models}
    if model_name not in installed_models and model_name not in model_aliases:
        raise RuntimeError(
            f'Ollama model "{model_name}" is not installed. '
            f'Run `ollama pull {model_name}` in a terminal, then restart the notebook kernel.'
        )


def get_embeddings_model():
    if LLM_PROVIDER == "openai":
        return OpenAIEmbeddings(
            model=OPENAI_EMBEDDING_MODEL,
            openai_api_key=_get_api_key("OPENAI_API_KEY"),
        )
    if LLM_PROVIDER == "ollama":
        ensure_ollama_model_available(OLLAMA_EMBEDDING_MODEL)
        return OllamaEmbeddings(
            model=OLLAMA_EMBEDDING_MODEL,
            base_url=OLLAMA_BASE_URL,
        )
    if LLM_PROVIDER == "gemini":
        embeddings = GoogleGenerativeAIEmbeddings(
            model=GEMINI_EMBEDDING_MODEL,
            google_api_key=_get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY"),
        )
        return GeminiEmbeddingsOneByOne(embeddings)


def get_chat_model():
    if LLM_PROVIDER == "openai":
        return ChatOpenAI(
            model=OPENAI_MODEL,
            openai_api_key=_get_api_key("OPENAI_API_KEY"),
        )
    if LLM_PROVIDER == "ollama":
        ensure_ollama_model_available(OLLAMA_MODEL)
        return ChatOllama(
            model=OLLAMA_MODEL,
            base_url=OLLAMA_BASE_URL,
        )
    if LLM_PROVIDER == "gemini":
        return ChatGoogleGenerativeAI(
            model=GEMINI_MODEL,
            google_api_key=_get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY"),
        )


def message_text(message):
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


print(f"Loaded env from: {PROJECT_ENV_PATH or 'not found'}")
print(f"Using {LLM_PROVIDER} chat model and matching embeddings")
if LLM_PROVIDER == "ollama":
    print(f"Ollama chat model: {OLLAMA_MODEL}")
elif LLM_PROVIDER == "gemini":
    print(f"Gemini chat model: {GEMINI_MODEL}")
print(f"LangSmith tracing: {os.getenv('LANGSMITH_TRACING', 'false')} | project: {os.getenv('LANGSMITH_PROJECT')}")


Loaded env from: /home/thimoty/git/building-llm-applications/.env
Using gemini chat model and matching embeddings
Gemini chat model: gemini-flash-latest
LangSmith tracing: true | project: qachatbot


# Self metadata query

## Ingestion (metadata enriched)

### Why the notebook starts with metadata-enriched ingestion

Before query generation can become more precise, the indexed content needs richer structure. The collection created here stores the same destination content as before, but each chunk is also tagged with metadata such as destination, region, and source URL. That makes it possible to combine semantic similarity with explicit filtering.

This setup matters because many real retrieval tasks are not just about finding similar text. They also need to restrict the search space to a place, time range, document source, or business entity that the user refers to in the question.


### Preparing the Chroma DB collection

In [2]:
from langchain_chroma import Chroma


In [3]:
uk_with_metadata_collection = Chroma(
    collection_name="uk_with_metadata_collection",
    embedding_function=get_embeddings_model())

uk_with_metadata_collection.reset_collection() #A
#A in case it already exists

### Defining content to be ingested and splitting strategy

In [4]:
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
html2text_transformer = Html2TextTransformer()

In [6]:
text_splitter = RecursiveCharacterTextSplitter( #A
    chunk_size=1000, chunk_overlap=100
)

In [7]:
def split_docs_into_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #B
    chunks = text_splitter.split_documents(
        text_docs)

    return chunks

In [8]:
uk_destinations = [
    ("Cornwall", "Cornwall"), ("North_Cornwall", "Cornwall"), 
    ("South_Cornwall", "Cornwall"), ("West_Cornwall", "Cornwall"),
    ("Tintagel", "Cornwall"), ("Bodmin", "Cornwall"), 
    ("Wadebridge", "Cornwall"),
    ("Penzance", "Cornwall"), ("Newquay", "Cornwall"), 
    ("St_Ives", "Cornwall"),
    ("Port_Isaac", "Cornwall"), ("Looe", "Cornwall"), 
    ("Polperro", "Cornwall"),
    ("Porthleven", "Cornwall"),
    ("East_Sussex", "East_Sussex"), ("Brighton", "East_Sussex"),
    ("Battle", "East_Sussex"), ("Hastings_(England)", "East_Sussex"),
    ("Rye_(England)", "East_Sussex"), ("Seaford", "East_Sussex"), 
    ("Ashdown_Forest", "East_Sussex")
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

In [9]:
uk_destination_url_with_metadata = [ #C 
    ( f'{wikivoyage_root_url}/{destination}', destination, region)
    for destination, region in uk_destinations]

#A Instantiate a relatively fine-chunk splitting strategy
#B Transform HTML docs into clean text docs
#C Prepare metadata to be imported: Url, UK Destination and UK Region

### Enriching a document with metadata: updating metadata

In [10]:
tintagel_url, tintagel_destination, tintagel_region = uk_destination_url_with_metadata[4]

In [11]:
tintagel_html_loader =AsyncHtmlLoader(tintagel_url)
tintagel_docs = tintagel_html_loader.load()

Fetching pages: 100%|##########| 1/1 [00:00<00:00, 10.17it/s]

In [12]:
# tintagel_docs # COMMENT: LangChain loaders create docs which contain metadata

In [13]:
for doc in tintagel_docs:
    doc.metadata['destination'] = tintagel_destination
    doc.metadata['region'] = tintagel_region
    print(doc.metadata)

{'source': 'https://en.wikivoyage.org/wiki/Tintagel', 'title': 'Tintagel – Travel guide at Wikivoyage', 'language': 'en', 'destination': 'Tintagel', 'region': 'Cornwall'}


### Enriching a document with metadata: creating metadata 

In [14]:
tintagel_docs_with_metadata = [
    Document(page_content=d.page_content,
             metadata = {
                 'source': tintagel_url,
                 'destination': tintagel_destination,
                 'region': tintagel_region
             })
    for d in tintagel_docs
]

In [15]:
# tintagel_docs_with_metadata # examine the Document

### Enriching the UK destination documents with metadata: creating metadata 

In [16]:
for (url, destination, region) in uk_destination_url_with_metadata:
    html_loader = AsyncHtmlLoader(url) #A
    docs =  html_loader.load() #B
    
    docs_with_metadata = [
        Document(page_content=d.page_content,
        metadata = {
            'source': url,
            'destination': destination,
            'region': region})
        for d in docs]
             
    chunks = split_docs_into_chunks(docs_with_metadata)

    print(f'Importing: {destination}')
    uk_with_metadata_collection.add_documents(documents=chunks)
#A Loader for one destination
#B Documents (chunks) related to one destination 

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  9.94it/s]


Importing: Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.11it/s]


Importing: North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 12.07it/s]


...

Importing: Rye_(England)


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 13.62it/s]


Importing: Seaford


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.49it/s]


Importing: Ashdown_Forest


## Q & A on a collection enriched with metadata

### Why metadata-enriched retrieval matters

Once chunks carry structured metadata, retrieval can combine two signals: semantic similarity over the chunk text and exact filtering over metadata fields. That lets the retriever answer questions like "festivals in Newquay" without searching across unrelated destinations that may contain similar wording.

The notebook shows three levels of sophistication: manual filters, automatic metadata inference with `SelfQueryRetriever`, and explicit structured generation with an LLM function call. All three rely on the same idea: the user question should be translated into both content search and retrieval constraints.


### Searching the collection with a metadata filter explicitly

In [17]:
question =  "Events or festivals"
metadata_retriever = uk_with_metadata_collection.as_retriever(
    search_kwargs={'k':2, 'filter':{'destination': 'Newquay'}})

result_docs = metadata_retriever.invoke(question)

In [18]:
result_docs

[Document(id='e053f6c1-0f4a-487c-924e-a5f22fb29318', metadata={'region': 'Cornwall', 'source': 'https://en.wikivoyage.org/wiki/Newquay', 'destination': 'Newquay'}, page_content="## Do\n\n[edit]\n\n  * Cornish Film Festival. Held annually for two weeks each November around Newquay.  (updated Jan 2024)\n  * 50.415741-5.0914781 Newquay Golf Club, Tower Road, TR7 1LT, ☏ +44 1637 872091, info@newquaygolfclub.co.uk. 9AM-4PM. A semi-private golf club established in 1890. Total yardage Championship: 6141, Men: 5708, and Women: 5364. £31 for non-members.  (updated Apr 2019)\n\n### Beaches\n\n[edit]\n\nFistral Beach\n\nNewquay is well known as a surfer's paradise. Therefore it offers plenty of\nbeaches:"),
 Document(id='4a0bd39c-7fb0-4920-90eb-62a1147d3d5e', metadata={'region': 'Cornwall', 'destination': 'Newquay', 'source': 'https://en.wikivoyage.org/wiki/Newquay'}, page_content='## See\n\n[edit]\n\n  * 50.414578-5.0848411 Blue Reef Aquarium, Towan Promenade, TR7 1DU (right next to Towan beach)

In [19]:
# COMMENT: As you can see, only chunks associated with'destination': 'Newquay' have been selected

### Why infer metadata filters from the question

Users rarely speak in the exact filter syntax expected by a retriever. A self-querying setup bridges that gap by turning a natural-language request into a richer internal query that contains both a semantic component and metadata restrictions.

This is useful because it preserves a natural user interface while still exploiting the structured fields stored during ingestion. In practice, it reduces the need for the application layer to hard-code filters for every possible phrasing of a question.


### Generating the self metadata query with the SelfQueryRetriever

In [20]:
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever #A
#A this requires pip install lark

In [21]:
metadata_field_info = [
    AttributeInfo(
        name="destination",
        description="The specific UK destination to be searched",
        type="string",
    ),
    AttributeInfo(
        name="region",
        description="The name of the UK region to be searched",
        type="string",
    )
]

In [22]:
question = "Tell me about events or festivals in the UK town of Newquay"

In [23]:
llm = get_chat_model()

self_query_retriever = SelfQueryRetriever.from_llm(
    llm, uk_with_metadata_collection, question, 
    metadata_field_info, verbose=True
)

In [24]:
result_docs = self_query_retriever.invoke(question)

In [25]:
result_docs

[Document(id='e053f6c1-0f4a-487c-924e-a5f22fb29318', metadata={'destination': 'Newquay', 'region': 'Cornwall', 'source': 'https://en.wikivoyage.org/wiki/Newquay'}, page_content="## Do\n\n[edit]\n\n  * Cornish Film Festival. Held annually for two weeks each November around Newquay.  (updated Jan 2024)\n  * 50.415741-5.0914781 Newquay Golf Club, Tower Road, TR7 1LT, ☏ +44 1637 872091, info@newquaygolfclub.co.uk. 9AM-4PM. A semi-private golf club established in 1890. Total yardage Championship: 6141, Men: 5708, and Women: 5364. £31 for non-members.  (updated Apr 2019)\n\n### Beaches\n\n[edit]\n\nFistral Beach\n\nNewquay is well known as a surfer's paradise. Therefore it offers plenty of\nbeaches:"),
 Document(id='d384f6c5-753b-4b85-b428-e3778828e90a', metadata={'region': 'Cornwall', 'source': 'https://en.wikivoyage.org/wiki/Newquay', 'destination': 'Newquay'}, page_content="* **Crantock Beach** \\- quiet beach, 2 km away from the city centre along the coastal path\n  * **Fistral Beach** \

### Generating the self metadata query with a LLM function call

#### Query schema

In [26]:
import datetime
from typing import Literal, Optional, Tuple, List

from pydantic import BaseModel, Field
from langchain_classic.chains.query_constructor.ir import (
    Comparator,
    Comparison,
    Operation,
    Operator,
    StructuredQuery,
)
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

In [27]:
class DestinationSearch(BaseModel):
    """Search over a vector database of tourist destinations."""

    content_search: str = Field(
        "",
        description="""Similarity search query applied 
        to tourist destinations.""",
    )
    destination: str = Field(
        ...,
        description="The specific UK destination to be searched.",
    )
    region: str = Field(
        ...,
        description="The name of the UK region to be searched.",
    )

    def pretty_print(self) -> None:
        for field in self.__fields__:
            if getattr(self, field) is not None and getattr(
                self, field) != getattr(
                self.__fields__[field], "default", None
            ):
                print(f"{field}: {getattr(self, field)}")

In [28]:
def build_filter(destination_search: DestinationSearch):
    comparisons = []

    destination = destination_search.destination #A
    region = destination_search.region #A
    
    if destination and destination != '': #B
        comparisons.append(
            Comparison(
                comparator=Comparator.EQ,
                attribute="destination",
                value=destination,
            )
        )
    if region and region != '': #C
        comparisons.append(
            Comparison(
                comparator=Comparator.EQ,
                attribute="region",
                value=region,
            )
        )    

    search_filter = Operation(operator=Operator.AND, 
                              arguments=comparisons) #D

    chroma_filter = ChromaTranslator().visit_operation(
        search_filter) #E
        
    return chroma_filter
#A Get destination and region from the structured query
#B If the destination exists, create an 'equality' operation
#C If the region exists, create an 'equality' operation
#D Create a combined search filter
#E Transform the filter into Chroma format

#### Conversion of user question to structured query including metadata filter

In [29]:
from langchain_core.prompts import ChatPromptTemplate

system_message = """You are an expert at converting user 
questions into vector database queries. 
You have access to a database of tourist destinations.
Given a question, return a database query optimized 
to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, 
do not try to rephrase them."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        ("human", "{question}"),
    ]
)
llm = get_chat_model()
structured_llm = llm.with_structured_output(
    DestinationSearch, method="function_calling")
query_generator = prompt | structured_llm

In [30]:
question = "Tell me about events or festivals in the UK town of Newquay"

structured_query =query_generator.invoke(question)

In [31]:
structured_query

DestinationSearch(content_search='events festivals', destination='Newquay', region='South West')

In [32]:
search_filter = build_filter(structured_query)

In [33]:
search_filter

{'$and': [{'destination': {'$eq': 'Newquay'}},
  {'region': {'$eq': 'South West'}}]}

In [34]:
search_query = structured_query.content_search

In [35]:
search_query

'events festivals'

In [36]:
metadata_retriever = uk_with_metadata_collection.as_retriever(
    search_kwargs={'k':3, 'filter': search_filter})

In [37]:
answer = metadata_retriever.invoke(search_query)

In [38]:
print(answer)

[]


In [39]:
## COMMENT: this is only the retrieval step; you still need to wrap it in a RAG chain

### Why generate SQL from natural language

A vector store is only one kind of backend. When the data lives in relational tables, retrieval requires SQL rather than semantic chunk search. The notebook therefore switches from vector retrieval to text-to-SQL, showing how a question about offers can be turned into a database query, cleaned, and then executed.

The important idea is that query generation must adapt to the storage model. The same user-facing interface can sit on top of both unstructured and structured data, but the generated query needs to match the capabilities and syntax of the target backend.


# Generating a structured SQL query

## Connecting to the UkBooking database

### Why the notebook initializes the SQLite database automatically

The SQL part of the notebook depends on a local SQLite file named `UkBooking.db`. In practice, that file may be missing, empty, or out of sync with the schema expected by the later cells. If the notebook tried to query the database immediately, those cases would fail with missing-table errors.

For that reason, the setup cell checks whether the database is usable before opening it through `SQLDatabase`. If the file is absent or does not contain the expected `Offer` table, the notebook rebuilds the database from `CreateUkBooking.sql` and `PopulateUkBooking.sql`. This keeps the SQL examples reproducible without requiring a manual setup step outside the notebook.


In [40]:
from langchain_community.utilities import SQLDatabase
from langchain_community.tools import QuerySQLDataBaseTool
from langchain_classic.chains import create_sql_query_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pathlib import Path
import sqlite3
import os


def resolve_ch10_dir():
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for directory in candidates:
        if (directory / "CreateUkBooking.sql").exists() and (directory / "PopulateUkBooking.sql").exists():
            return directory
        if (directory / "ch10" / "CreateUkBooking.sql").exists() and (directory / "ch10" / "PopulateUkBooking.sql").exists():
            return directory / "ch10"
    raise FileNotFoundError("Could not locate the ch10 SQL setup files.")


CH10_DIR = resolve_ch10_dir()
UK_BOOKING_DB_PATH = CH10_DIR / "UkBooking.db"
SCHEMA_SQL_PATH = CH10_DIR / "CreateUkBooking.sql"
SEED_SQL_PATH = CH10_DIR / "PopulateUkBooking.sql"


def ensure_uk_booking_db_ready():
    needs_init = not UK_BOOKING_DB_PATH.exists() or UK_BOOKING_DB_PATH.stat().st_size == 0

    if not needs_init:
        conn = sqlite3.connect(UK_BOOKING_DB_PATH)
        try:
            offer_exists = conn.execute(
                "SELECT name FROM sqlite_master WHERE type='table' AND name='Offer'"
            ).fetchone()
            needs_init = offer_exists is None
        finally:
            conn.close()

    if not needs_init:
        return UK_BOOKING_DB_PATH

    if UK_BOOKING_DB_PATH.exists():
        UK_BOOKING_DB_PATH.unlink()

    conn = sqlite3.connect(UK_BOOKING_DB_PATH)
    try:
        conn.executescript(SCHEMA_SQL_PATH.read_text(encoding="utf-8"))
        conn.executescript(SEED_SQL_PATH.read_text(encoding="utf-8"))
        conn.commit()
    finally:
        conn.close()

    return UK_BOOKING_DB_PATH


In [41]:
db_path = ensure_uk_booking_db_ready()
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
print(f"Using database: {db_path}")
print(db.get_usable_table_names())


Using database: /home/thimoty/git/building-llm-applications/ch10/UkBooking.db
['Accommodation', 'AccommodationType', 'Booking', 'Customer', 'Destination', 'Offer']


In [42]:
db.run("SELECT * FROM Offer;")

"[(1, 1, 'Summer Special', 0.15, '2024-06-01', '2024-08-31'), (2, 2, 'Weekend Getaway', 0.1, '2024-09-01', '2024-12-31'), (3, 3, 'Early Bird Discount', 0.2, '2024-05-01', '2024-06-30'), (4, 4, 'Stay 3 Nights, Get 1 Free', 0.25, '2024-01-01', '2024-03-31'), (5, 5, 'Historic Stay Offer', 0.1, '2024-04-01', '2024-06-30'), (6, 6, 'Autumn Discount', 0.15, '2024-09-01', '2024-11-30'), (7, 7, 'Cottage Retreat Offer', 0.12, '2024-07-01', '2024-09-30'), (8, 8, 'City Break Deal', 0.08, '2024-10-01', '2024-12-31'), (9, 9, 'Luxury Villa Offer', 0.18, '2024-05-01', '2024-08-31'), (10, 10, 'Spa & Wellness Package', 0.2, '2024-04-01', '2024-07-31')]"

## Generate SQL queries from natural language

### Generating the SQL query

In [43]:
llm = get_chat_model()
sql_query_gen_chain = create_sql_query_chain(llm, db)
response = sql_query_gen_chain.invoke(
    {"question": 
     "Give me some offers for Cardiff, including the hotel name"})

In [44]:
response

'SQLQuery: SELECT "T1"."OfferDescription", "T2"."Name" FROM "Offer" AS "T1" JOIN "Accommodation" AS "T2" ON "T1"."AccommodationId" = "T2"."AccommodationId" JOIN "Destination" AS "T3" ON "T2"."DestinationId" = "T3"."DestinationId" WHERE "T3"."Name" = \'Cardiff\' LIMIT 5;'

In [45]:
#db.run(response) # returns error

### Executing the SQL query [NOTE: THIS WILL THROW AN ERROR]

In [46]:
sql_query_exec_chain = QuerySQLDataBaseTool(db=db)
sql_query_gen_chain = create_sql_query_chain(llm, db)
chain = sql_query_gen_chain | sql_query_exec_chain
chain.invoke({"question": "Give me some offers for Cardiff, including the hotel name"})

/tmp/ipykernel_252718/2637313701.py:1: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  sql_query_exec_chain = QuerySQLDataBaseTool(db=db)


'Error: (sqlite3.OperationalError) near "SQLQuery": syntax error\n[SQL: SQLQuery: SELECT "T2"."Name", "T1"."OfferDescription", "T1"."DiscountRate" FROM "Offer" AS "T1" JOIN "Accommodation" AS "T2" ON "T1"."AccommodationId" = "T2"."AccommodationId" JOIN "Destination" AS "T3" ON "T2"."DestinationId" = "T3"."DestinationId" WHERE "T3"."Name" = \'Cardiff\' LIMIT 5;]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)'

### Fixing the SQL format

In [47]:
clean_sql_prompt_template = """You are an expert in SQL Lite. 
You are asked to fix badly formed SQL Lite queries, 
which might contain unneded prefixes or suffixes. 
Given the following unclean SQL statement, 
transform it to a clean, 
executable SQL statement for SQL lite.
Always prefix column names with the table name.
Only return an executable SQL statement which terminates 
with a semicolon. Do not return anything else.
Do not include the language name or symbols like ```.

Unclean SQL: {unclean_sql}"""

In [48]:
clean_sql_prompt = ChatPromptTemplate.from_template(
    clean_sql_prompt_template)

In [49]:
clean_sql_chain = clean_sql_prompt | llm

In [50]:
full_sql_gen_chain = sql_query_gen_chain | \
   clean_sql_chain | StrOutputParser()

In [51]:
question = """Give me some offers for Cardiff, 
including the accomodation name"""

In [52]:
response = full_sql_gen_chain.invoke({"question": question})

In [53]:
response

'SELECT "Offer"."OfferDescription", "Accommodation"."Name" FROM "Offer" JOIN "Accommodation" ON "Offer"."AccommodationId" = "Accommodation"."AccommodationId" JOIN "Destination" ON "Accommodation"."DestinationId" = "Destination"."DestinationId" WHERE "Destination"."Name" = \'Cardiff\' LIMIT 5;'

In [54]:
### Comment: now SQL is fixed

### Executing the SQL query

In [55]:
sql_query_exec_chain = QuerySQLDataBaseTool(db=db)

In [56]:
sql_query_gen_and_exec_chain = full_sql_gen_chain \
    | sql_query_exec_chain | StrOutputParser()

In [57]:
response = sql_query_gen_and_exec_chain.invoke(
    {"question":question})

In [58]:
response

"[('Early Bird Discount', 0.2, '2024-05-01', '2024-06-30', 'Cardiff Camping')]"

In [59]:
## COMMENT: this is only the retrieval step; you still need to wrap it in a RAG chain

### Why route questions across data sources

A larger RAG system often has access to more than one store. Tourist descriptions fit naturally in a vector collection, while booking offers belong in a relational database. A router sits in front of those retrievers and decides which path should handle a given question.

This makes the application more robust and scalable. Instead of forcing every question through the same retrieval method, the system can classify intent first and then send the request to the backend most likely to return relevant context.


# Query router

In [60]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableLambda

## Setting up the data retrievers

### Setting up the vector store retriever

In [61]:
tourist_info_retriever_chain = RunnableLambda(
    lambda x: x['question']) \
       | uk_with_metadata_collection.as_retriever(
           search_kwargs={'k':2}) 

### Setting up the relational database retriever (Same as sql_query_gen_and_exec_chain above)

In [62]:
uk_accommodation_retriever_chain =  full_sql_gen_chain \
    | sql_query_exec_chain | StrOutputParser()

## Setting up the query router

In [63]:
class RouteQuery(BaseModel):
    """Route a user question to the most relevant datasource."""

    datasource: Literal["tourist_info_store", 
        "uk_booking_db"] = Field(
        ...,
        description="""Given a user question, 
        route it either to a tourist info vector store 
        or a UK accomodation booking relational database.""",
    )

llm = get_chat_model()
structured_llm_router = llm.with_structured_output(
    RouteQuery) #A
#A Structured router which uses LLM function calls

### Setting up the question router chain

In [64]:
system = """You are an expert at routing a user question 
to a tourist info vector store 
or to an UK accommodation booking relational database.
The vector store contains tourist information about UK destinations.
Use the vectorstore for general tourist information questions 
on UK destinations. 
For questions about accommodation availability or booking, 
use the UK Booking database."""
route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

### Testing the router chain

In [65]:
selected_data_source = question_router.invoke(
    {"question": "Have you got any offers in Brighton?"}
)

In [66]:
print(selected_data_source)

datasource='uk_booking_db'


In [67]:
selected_data_source = question_router.invoke(
    {"question": "Where are the best beaches in Cornwall?"}
)

In [68]:
print(selected_data_source)

datasource='tourist_info_store'


### Setting up the retriever chooser

In [69]:
retriever_chains = {
    'tourist_info_store': tourist_info_retriever_chain,
    'uk_booking_db': uk_accommodation_retriever_chain
}

def retriever_chooser(question):
    selected_data_source = question_router.invoke(
        {"question": question})

    return retriever_chains[selected_data_source.datasource]

In [70]:
chosen = retriever_chooser("""Tell me about events 
or festivals in the UK town of Newquay""") 

In [71]:
print(chosen)

first=RunnableLambda(...) middle=[] last=VectorStoreRetriever(tags=['Chroma', 'GeminiEmbeddingsOneByOne'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7366daf4aea0>, search_kwargs={'k': 2})


## Setting up the full RAG chain

In [72]:
from langchain_core.runnables import RunnablePassthrough

In [73]:
rag_prompt_template = """
Given a question and some context, answer the question.
If you get a structured context, like a tuple, try to 
infer the meaning of the components: 
typically they refer to accommodation offers, 
and the number is a percentage (0.2 means 20%).
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template) 

def execute_rag_chain(question, chosen_retriever):
    full_rag_chain = (
        {
            "context": {"question": RunnablePassthrough()} 
                | chosen_retriever,#A
            "question": RunnablePassthrough(),#B
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )

    return full_rag_chain.invoke(question)

#A The context is returned by the retriver after feeding to it the rewritten query
#B This is the original user question

## Executing the full RAG chain 

### Question on accommodation offers

In [74]:
question = """Give me some offers for Cardiff, 
including the accommodation name"""

chosen_retriever = retriever_chooser(question)

answer = execute_rag_chain(question, chosen_retriever)

In [75]:
print(answer)

At Cardiff Camping, you can get an Early Bird Discount of 20%.


### Question on tourist information

In [76]:
question_2 = """Tell me about events or festivals 
in the UK town of Newquay"""

chosen_retriever_2 = retriever_chooser(question_2)

answer2 = execute_rag_chain(question_2, chosen_retriever_2)

In [77]:
print(answer2)

Newquay hosts the **Cornish Film Festival**, which is held annually for two weeks each November.


### Why apply retrieval postprocessing

Even after a good query has been generated, the retrieved context may still be incomplete or unevenly ranked. Retrieval postprocessing addresses that problem by combining multiple retrieval runs and reranking the merged results.

The notebook uses RAG Fusion with Reciprocal Rank Fusion to aggregate results from several query variations. This helps recover relevant chunks that a single retrieval pass might miss, especially when the question can be phrased from several valid angles.


# Retrieval post processing

## RAG Fusion

### Multiple query Generation (same as for MultiQueryRetriver)

In [78]:
from langchain_core.prompts import ChatPromptTemplate

from typing import List
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

In [79]:
multi_query_gen_prompt_template = """
You are an AI language model assistant. Your task is 
to generate five different versions of the given user 
question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the 
user question, your goal is to help
the user overcome some of the limitations of the 
distance-based similarity search. 
Provide these alternative questions separated by newlines.
Original question: {question}
"""

multi_query_gen_prompt = ChatPromptTemplate.from_template(
    multi_query_gen_prompt_template) 

In [80]:
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Parse out a question from each output line."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return list(filter(None, lines))  


questions_parser = LineListOutputParser()

In [81]:
llm = get_chat_model()

In [82]:
multi_query_gen_chain = multi_query_gen_prompt | llm | questions_parser

### Reciprocal Rank Fusion algorithm

In [83]:
# Based on: https://github.com/Raudaschl/rag-fusion/blob/master/main.py

def reciprocal_rank_fusion(results_groups: #A
                           list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple groups of 
        ranked documents and an optional parameter k used in 
        the Reciprocal Rank Fusion (RRF) formula """

    indexed_results = {} #B
    
    for group_id, results_group in enumerate(
        results_groups): #V
        for local_rank, doc in enumerate(results_group):
            indexed_results[(group_id, local_rank)] = doc
    
    fused_scores = {} #D
    
    for key, doc in indexed_results.items(): #E
        group_id, local_rank = key

        if key not in fused_scores:
            fused_scores[key] = 0 #F
        
        doc_current_score = fused_scores[key]        
        fused_scores[key] += 1 / (local_rank + k) #G

    reranked_results = [ #H
        (indexed_results[key], score)
        for key, score in sorted(fused_scores.items(), 
                                 key=lambda x: x[1], reverse=True)
    ]

    return reranked_results
#A Based on: https://github.com/Raudaschl/rag-fusion/blob/master/main.py                
# B Initialize a dictionary to organize results with an index
# C Index the results by (group_id, local_rank)
# D Initialize a dictionary to hold fused scores for each unique document
# E Iterate through the indexed results
# F Initialize an indexed result with a score of 0 if it has not been processed yet
# G calculate the new document score with the RRF formula
# H rerank the results by RRF score 

In [84]:
retriever = uk_with_metadata_collection.as_retriever(
    search_kwargs={'k':3})
top_three_results = RunnableLambda(
    lambda x: x[0:3]) #A

rag_fusion_retrieval_chain =multi_query_gen_chain \
    | retriever.map() | reciprocal_rank_fusion \
    | top_three_results #B
        
docs = rag_fusion_retrieval_chain.invoke(
    {"question": question}) #C
len(docs)
#A select the top three results
#B Full RAG fusion retrieval chain
#C testing the retrieval_chain_rag_fusion chain

3

### Incorporating Rag Fusion into the RAG Chain

In [85]:
rag_prompt_template = """
Given a question and some context, answer the question.
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template) 

rag_chain = (
    {
        "context": {"question": RunnablePassthrough()} | rag_fusion_retrieval_chain,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the step-back question
#B This is the original user question

In [86]:
user_question = "Can you give me some tips for a trip to Brighton?"

answer = rag_chain.invoke(user_question)
print(answer)

I do not know. The provided context is a table of contents and does not contain specific tips or descriptive information about Brighton.


In [87]:
# Test model on nvidia dev portal
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = "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"
)

completion = client.chat.completions.create(
  model="deepseek-ai/deepseek-v3.2",
  messages=[{"role":"user","content":"Write a short poem about Eirini in modern greek"}],
  temperature=1,
  top_p=0.95,
  max_tokens=8192,
  extra_body={"chat_template_kwargs": {"thinking":True}},
  stream=True
)

for chunk in completion:
  if not getattr(chunk, "choices", None):
    continue
  reasoning = getattr(chunk.choices[0].delta, "reasoning_content", None)
  if reasoning:
    print(reasoning, end="")
  if chunk.choices and chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")
  



KeyboardInterrupt: 